# 00. Environment and code setup
Run setup first. Use the same Drive project folder. Run one stage cell at a time and share its output for review. Never use Run all across stages.


## 1. Mount Drive
Authorize the account containing the existing project data. No temporary-storage fallback is used.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, subprocess, json, hashlib, csv
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/FoodPriceWatch')
assert Path('/content/drive/MyDrive').is_dir(), 'STOP: Drive is not mounted'
print('Project root:', ROOT)
print('Existing project folder:', ROOT.exists())


## 2. Load code
Updates preserve Drive stage status and local stage commits, including when histories diverge. A backup branch is created before merging. Unexpected conflicts stop the update and undo the merge. Uncommitted changes need agent review.


In [ ]:
"""Update a Drive checkout while preserving its committed stage state."""
import json
from pathlib import Path
import subprocess
import time

STATUS_FIELDS = {'status', 'started_at', 'last_unit', 'units_done',
                 'output_sha256', 'git_commit', 'finished_at', 'notes'}


def sync(root):
    root = Path(root)

    def git(*args, check=True):
        result = subprocess.run(['git', *args], cwd=root, text=True,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        if result.stdout.strip():
            print(result.stdout.strip())
        if check and result.returncode:
            raise RuntimeError('STOP: git ' + ' '.join(args) + ' failed; see output above')
        return result

    assert not git('status', '--porcelain').stdout.strip(), 'STOP: uncommitted changes; send git status to the agent'
    assert git('branch', '--show-current').stdout.strip() == 'main', 'STOP: expected main branch'
    for name in ['MERGE_HEAD', 'rebase-merge', 'rebase-apply']:
        path = git('rev-parse', '--git-path', name).stdout.strip()
        assert not (root / path).exists(), 'STOP: unfinished Git operation'
    git('fetch', 'origin', 'main')
    # Local runtime status is authoritative; GitHub carries code, not Drive evidence.
    local = json.loads((root / 'state/stages.json').read_text())
    remote = json.loads(git('show', 'origin/main:state/stages.json').stdout)
    by_id = {s['id']: s for s in remote['stages']}
    for stage in local['stages']:
        assert stage['id'] in by_id, 'STOP: upstream removed a stage'
        other = by_id[stage['id']]
        assert {k: v for k, v in stage.items() if k not in STATUS_FIELDS} == {
            k: v for k, v in other.items() if k not in STATUS_FIELDS
        }, 'STOP: stage definition changed; agent review required: ' + stage['id']
        for key in STATUS_FIELDS:
            if key in stage:
                other[key] = stage[key]
    backup = 'backup/colab-before-sync-' + str(time.time_ns())
    git('branch', backup)
    print('Previous committed state preserved at:', backup)
    git('config', 'user.name', 'FPW runner')
    git('config', 'user.email', 'fpw@example.com')
    result = git('merge', '--no-ff', '--no-commit', 'origin/main', check=False)
    conflicts = git('diff', '--name-only', '--diff-filter=U').stdout.splitlines()
    if result.returncode and (not conflicts or set(conflicts) - {'state/stages.json'}):
        git('merge', '--abort', check=False)
        raise RuntimeError('STOP: update conflict; merge undone, previous commits retained. Send the output above to the agent')
    ledger = root / 'state/stages.json'
    # Write only when content changed, using an atomic replacement.
    if conflicts or json.loads(ledger.read_text()) != remote:
        temporary = ledger.with_suffix('.sync.tmp')
        temporary.write_text(json.dumps(remote, indent=2) + '\n')
        temporary.replace(ledger)
    git('add', 'state/stages.json')
    merge_head = root / git('rev-parse', '--git-path', 'MERGE_HEAD').stdout.strip()
    if merge_head.exists() or git('diff', '--cached', '--quiet', check=False).returncode:
        git('commit', '-m', 'Sync project code while preserving Drive stage state')
    print('Code update complete. Runtime data and prior commits retained.')
    git('log', '--oneline', '-3')



REPO = 'https://github.com/mrchowdhury587/foodprice_watch.git'
def git(*args):
    return subprocess.check_output(['git', *args], cwd=ROOT, text=True).strip()
if not (ROOT / '.git').exists():
    assert not ROOT.exists() or not any(ROOT.iterdir()), 'STOP: existing non-Git folder; ask the agent to attach code without replacing data'
    ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', REPO, str(ROOT)], check=True)
else:
    sync(ROOT)
for name in ['src/fpw/features.py', 'src/stages/s23_features.py']:
    assert (ROOT / name).is_file(), 'STOP: push merged feature code to GitHub first: ' + name
subprocess.run(['git', 'config', 'user.name', 'FPW runner'], cwd=ROOT, check=True)
subprocess.run(['git', 'config', 'user.email', 'fpw@example.com'], cwd=ROOT, check=True)
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'src')
sys.path.insert(0, str(ROOT / 'src'))
print('Code commit:', git('rev-parse', 'HEAD'))


## 3. Install runtime packages


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements-colab.txt')], check=True)
os.environ.setdefault('FPW_THREADS', str(os.cpu_count() or 2))
print('Package install: PASS; threads:', os.environ['FPW_THREADS'])


## 4. Startup verification
A failed check stops execution. Missing or changed finished outputs are reset to pending by the project checker; paste failures back before continuing.


In [ ]:
for folder in ['outputs', 'data']:
    for tmp in (ROOT / folder).rglob('*.tmp'):
        assert tmp.resolve().is_relative_to(ROOT.resolve())
        tmp.unlink()
        print('Removed interrupted-write temporary file:', tmp.relative_to(ROOT))
subprocess.run([sys.executable, '-m', 'fpw.check'], cwd=ROOT, check=True)
def run_unit(sid):
    subprocess.run([sys.executable, '-m', 'fpw.runner', sid], cwd=ROOT, check=True)
    ledger = json.loads((ROOT / 'state/stages.json').read_text())
    stage = next(s for s in ledger['stages'] if s['id'] == sid)
    assert stage['status'] == 'done', 'STOP: stage was not finalized'
    summary = []
    for name, expected in stage['output_sha256'].items():
        path = ROOT / name
        actual = hashlib.sha256(path.read_bytes()).hexdigest()
        assert actual == expected, 'STOP: output hash mismatch: ' + name
        rows = None
        if path.suffix == '.csv':
            with path.open(newline='') as f:
                rows = sum(1 for _ in csv.DictReader(f))
        elif path.suffix == '.parquet':
            import pyarrow.parquet as pq
            rows = pq.read_metadata(path).num_rows
        summary.append(dict(file=name, rows=rows, sha256=actual))
    assert summary, 'STOP: no finalized output hashes'
    print(json.dumps(dict(unit=sid, outputs=summary, check='PASS', git_commit=git('rev-parse', 'HEAD')), indent=2))
